In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone variable as required (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into DataFrames (cache for later use)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# 1) For each file: total row count, min timestamp, max timestamp
def file_summary(df, name):
    return {
        "file": name,
        "rows": int(df.shape[0]),
        "min_timestamp": int(df["timestamp"].min()) if not df.empty else None,
        "max_timestamp": int(df["timestamp"].max()) if not df.empty else None
    }

file_summaries = [
    file_summary(metric_df, "metric.csv"),
    file_summary(trace_df, "trace.csv"),
    file_summary(log_df, "log.csv"),
    file_summary(error_logs_df, "error_logs.csv")
]
file_summary_df = pd.DataFrame(file_summaries)

# Utility to compute group-by count and global P95 for a given df and key columns
def group_count_p95(df, group_cols, value_col="value", top_n=20):
    # Ensure value column is numeric
    df_loc = df.copy()
    df_loc[value_col] = pd.to_numeric(df_loc[value_col], errors='coerce')
    agg = (
        df_loc
        .groupby(group_cols, as_index=False)[value_col]
        .agg(count="count")
    )
    # compute p95 per group using groupby apply for clarity (works with NaNs)
    p95_series = (
        df_loc
        .groupby(group_cols)[value_col]
        .apply(lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan)
        .reset_index(name="p95_value")
    )
    result = agg.merge(p95_series, on=group_cols)
    result = result.sort_values("count", ascending=False).reset_index(drop=True)
    return result.head(top_n)

# 2) metric.csv grouping
metric_top20 = group_count_p95(metric_df, ["cmdb_id", "kpi_name"], value_col="value", top_n=20)

# 3) trace.csv grouping
trace_top20 = group_count_p95(trace_df, ["cmdb_id", "trace_name"], value_col="value", top_n=20)

# 4) log.csv grouping
log_top20 = group_count_p95(log_df, ["cmdb_id", "log_name"], value_col="value", top_n=20)

# 5) error_logs.csv: total rows/min/max already in file_summary_df; also counts per cmdb_id (top20)
error_counts_df = (
    error_logs_df
    .groupby("cmdb_id", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
    .head(20)
)

# top 20 rows from error_logs (compact)
error_top20 = error_logs_df[["timestamp", "cmdb_id", "message"]].head(20)

# Final displayed outputs (compact aggregates only)
file_summary_df, metric_top20, trace_top20, log_top20, error_counts_df, error_top20

```
Out[1]:
```
```python
# Summarize the previously computed aggregates in plain English.
# Uses cached variables from prior run: file_summary_df, metric_top20, trace_top20, log_top20, error_counts_df, error_top20

def safe_int(x):
    try:
        return int(x)
    except:
        return x

# File-level summaries
fs = {row['file']: row for _, row in file_summary_df.iterrows()}
metric_rows = safe_int(fs.get("metric.csv", {}).get("rows"))
metric_min = fs.get("metric.csv", {}).get("min_timestamp")
metric_max = fs.get("metric.csv", {}).get("max_timestamp")
trace_rows = safe_int(fs.get("trace.csv", {}).get("rows"))
trace_min = fs.get("trace.csv", {}).get("min_timestamp")
trace_max = fs.get("trace.csv", {}).get("max_timestamp")
log_rows = safe_int(fs.get("log.csv", {}).get("rows"))
log_min = fs.get("log.csv", {}).get("min_timestamp")
log_max = fs.get("log.csv", {}).get("max_timestamp")
error_rows = safe_int(fs.get("error_logs.csv", {}).get("rows"))
error_min = fs.get("error_logs.csv", {}).get("min_timestamp")
error_max = fs.get("error_logs.csv", {}).get("max_timestamp")

# Helper to format top N group examples
def top_examples(df, cols, n=3):
    examples = []
    for i, row in df.head(n).iterrows():
        items = ", ".join(f"{c}={row[c]}" for c in cols)
        examples.append(f"[{i}] {items}")
    return examples

metric_examples = top_examples(metric_top20, ["cmdb_id", "kpi_name", "count", "p95_value"], n=3)
trace_examples = top_examples(trace_top20, ["cmdb_id", "trace_name", "count", "p95_value"], n=3)
log_examples = top_examples(log_top20, ["cmdb_id", "log_name", "count", "p95_value"], n=3)

# Build the summary string (concise)
summary = (
    "Summary of telemetry files (compact):\n\n"
    f"1) File-level counts and timestamp ranges:\n"
    f"- metric.csv: rows={metric_rows}, min_timestamp={metric_min}, max_timestamp={metric_max}\n"
    f"- trace.csv:  rows={trace_rows}, min_timestamp={trace_min}, max_timestamp={trace_max}\n"
    f"- log.csv:    rows={log_rows}, min_timestamp={log_min}, max_timestamp={log_max}\n"
    f"- error_logs.csv: rows={error_rows}, min_timestamp={error_min}, max_timestamp={error_max}\n\n"
    "2) metric.csv (top 20 groups by row count):\n"
    "   Observation: Top groups are dominated by Tomcat04 (many groups with count=31).\n"
    "   Examples:\n" + "\n".join("    - " + e for e in metric_examples) + "\n\n"
    "3) trace.csv (top 20 groups by row count):\n"
    "   Observation: Top groups include IG01, dockerA2, MG01 with count=31 for many groups.\n"
    "   Examples:\n" + "\n".join("    - " + e for e in trace_examples) + "\n\n"
    "4) log.csv (top 20 groups by row count):\n"
    "   Observation: apache02, Tomcat03, Tomcat02, Tomcat04 appear among top groups (many with count=31).\n"
    "   Examples:\n" + "\n".join("    - " + e for e in log_examples) + "\n\n"
    "5) error_logs.csv:\n"
    "   Observation: No records (total rows = 0). Therefore no per-cmdb_id counts or top rows to show.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file   rows  min_timestamp  max_timestamp
0      metric.csv  29675   1.616602e+09   1.616603e+09
1       trace.csv   6915   1.616602e+09   1.616603e+09
2         log.csv    397   1.616602e+09   1.616603e+09
3  error_logs.csv      0            NaN            NaN,      cmdb_id                                           kpi_name  count     p95_value
0   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
1   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
2   Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00
3   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+02
4   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+00
5   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02
6   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00
7   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00
8   Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00
9   Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00
10  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00
11  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
12  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
13  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00
14  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  7.000000e+02
15  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  4.136860e+05
16  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  1.325676e+09
17  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2.276624e+06
18  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00
19  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00,      cmdb_id                         trace_name  count   p95_value
0       IG01           trace.self.duration_mean     31    0.375121
1       IG01            trace.self.duration_p95     31    1.695625
2   dockerA2      trace.from_MG02.duration_mean     31    0.110978
3       IG01               trace.self.row_count     31  606.000000
4   dockerA2           trace.self.duration_mean     31    0.013142
5   dockerA2          trace.from_MG02.row_count     31   55.000000
6   dockerA2               trace.self.row_count     31  821.000000
7   dockerA2       trace.from_MG02.duration_p95     31    0.611925
8   dockerA2            trace.self.duration_p95     31    0.008500
9       MG01   trace.from_Tomcat03.duration_p95     31    1.186675
10      MG01           trace.self.duration_mean     31    0.165678
11      MG01      trace.from_Tomcat03.row_count     31  110.500000
12      MG01  trace.from_Tomcat03.duration_mean     31    0.277168
13      MG01            trace.self.duration_p95     31    1.058350
14      MG01               trace.self.row_count     31  627.500000
15  dockerA1               trace.self.row_count     31  798.500000
16  dockerA1            trace.self.duration_p95     31    0.008000
17  dockerA1           trace.self.duration_mean     31    0.012071
18      MG02      trace.from_Tomcat03.row_count     31   77.500000
19      MG02   trace.from_Tomcat03.duration_p95     31    1.665650,      cmdb_id          log_name  count  p95_value
0   apache02   log.error_count     31      0.000
1   Tomcat03   log.error_count     31      0.000
2   Tomcat02     log.row_count     31    156.000
3   Tomcat02   log.error_count     31      0.000
4   apache02     log.row_count     31    297.500
5   apache01     log.row_count     31    297.000
6   Tomcat04   log.error_count     31      0.000
7   Tomcat03     log.row_count     31    156.000
8   apache01   log.error_count     31      0.000
9   Tomcat04     log.row_count     31    159.500
10  Tomcat01   log.error_count     30      0.000
11  Tomcat01     log.row_count     30    140.850
12  Tomcat04  log.gc_pause_max      7      0.090
13  Tomcat03  log.gc_pause_max      6      0.085
14  Tomcat02  log.gc_pause_max      5      0.080
15  Tomcat01  log.gc_pause_max      5      0.082
16   Mysql02     log.row_count      1      1.000
17   Mysql02   log.error_count      1      0.000
18   Mysql01     log.row_count      1      1.000
19   Mysql01   log.error_count      1      0.000, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[2]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# ensure timezone (UTC+8) variable available per rules
tz = pytz.timezone('Asia/Shanghai')

# reuse cached metric_df if present, otherwise load from CSV
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Candidate components list
components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat04","Tomcat03",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# Filter to candidate components (do not filter by any time window)
metric_candidates = metric_df[metric_df["cmdb_id"].isin(components)].copy()

# Ensure numeric values
metric_candidates["value"] = pd.to_numeric(metric_candidates["value"], errors="coerce")

# Group and compute count, global P95 and P5 for each (cmdb_id, kpi_name)
gb = metric_candidates.groupby(["cmdb_id", "kpi_name"])

count_df = gb["value"].count().reset_index(name="count_full_series")

p95_df = (
    gb["value"]
    .apply(lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size > 0 else np.nan)
    .reset_index(name="p95_value")
)

p5_df = (
    gb["value"]
    .apply(lambda x: float(np.nanpercentile(x.dropna(), 5)) if x.dropna().size > 0 else np.nan)
    .reset_index(name="p5_value")
)

# Merge into final thresholds table and cache it
thresholds_df = count_df.merge(p95_df, on=["cmdb_id", "kpi_name"]).merge(p5_df, on=["cmdb_id", "kpi_name"])
thresholds_df = thresholds_df.sort_values("count_full_series", ascending=False).reset_index(drop=True)

# Compact top-20 display
thresholds_df_top20 = thresholds_df[["cmdb_id", "kpi_name", "count_full_series", "p95_value", "p5_value"]].head(20)

# Return the compact top-20 table (thresholds_df is cached in kernel)
thresholds_df_top20

```
Out[3]:
```
```python
# Summarize the computed thresholds (plain English) using cached thresholds_df
total_groups = int(thresholds_df.shape[0])
top_sample = thresholds_df.head(3)[["cmdb_id", "kpi_name", "count_full_series", "p95_value", "p5_value"]]

summary = (
    f"Computed global thresholds for candidate components. Total (cmdb_id, kpi_name) groups examined: {total_groups}.\n\n"
    "Top observations (based on highest full-series counts):\n"
    f"- The top 20 groups shown are dominated by Mysql02; each of these top groups has count_full_series = 31.\n"
    "- Example top 3 groups:\n"
    f"  1) {top_sample.iloc[0].cmdb_id} | {top_sample.iloc[0].kpi_name} | count={int(top_sample.iloc[0].count_full_series)} | p95={top_sample.iloc[0].p95_value} | p5={top_sample.iloc[0].p5_value}\n"
    f"  2) {top_sample.iloc[1].cmdb_id} | {top_sample.iloc[1].kpi_name} | count={int(top_sample.iloc[1].count_full_series)} | p95={top_sample.iloc[1].p95_value} | p5={top_sample.iloc[1].p5_value}\n"
    f"  3) {top_sample.iloc[2].cmdb_id} | {top_sample.iloc[2].kpi_name} | count={int(top_sample.iloc[2].count_full_series)} | p95={top_sample.iloc[2].p95_value} | p5={top_sample.iloc[2].p5_value}\n\n"
    "Note: Many p5 values are 0.0 for these Mysql02 metrics, and p95 values vary widely (some very large), indicating skewed distributions for those KPIs.\n\n"
    "The full thresholds table is cached in the kernel variable 'thresholds_df' for subsequent steps."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    kpi_name  count_full_series    p95_value      p5_value
0   Mysql02                Mysql-MySQL_3306_Select Scan                 31   35709464.5  2.991650e+00
1   Mysql02     Mysql-MySQL_3306_Slave Open Temp Tables                 31          0.0  0.000000e+00
2   Mysql02        Mysql-MySQL_3306_Slow launch threads                 31          0.0  0.000000e+00
3   Mysql02                Mysql-MySQL_3306_SlowQueries                 31   16974237.0  7.535500e-01
4   Mysql02      Mysql-MySQL_3306_Table Locks Immediate                 31     387871.0  6.615000e-02
5   Mysql02         Mysql-MySQL_3306_Table Locks Waited                 31          0.0  0.000000e+00
6   Mysql02      Mysql-MySQL_3306_Table open cache hits                 31  555366316.0  5.373305e+01
7   Mysql02    Mysql-MySQL_3306_Table open cache misses                 31      38210.0  0.000000e+00
8   Mysql02           Mysql-MySQL_3306_Select Full Join                 31          2.0  0.000000e+00
9   Mysql02     Mysql-MySQL_3306_Select Full Range Join                 31          0.0  0.000000e+00
10  Mysql02               Mysql-MySQL_3306_Select Range                 31         13.0  0.000000e+00
11  Mysql02         Mysql-MySQL_3306_Select Range Check                 31          0.0  0.000000e+00
12  Mysql02            Mysql-MySQL_3306_Threads Created                 31         37.0  0.000000e+00
13  Mysql02           Mysql-MySQL_3306_ThreadsConnected                 31         22.0  2.100000e+01
14  Mysql02             Mysql-MySQL_3306_ThreadsRunning                 31          8.5  1.000000e+00
15  Mysql02  Mysql-MySQL_3306_max trx lock memory bytes                 31       1136.0  0.000000e+00
16  Mysql02          Mysql-MySQL_3306_Sort Merge Passes                 31          6.0  0.000000e+00
17  Mysql02                 Mysql-MySQL_3306_Sort Range                 31      43914.0  0.000000e+00
18  Mysql02                  Mysql-MySQL_3306_Sort Rows                 31    1210387.0  1.209864e+06
19  Mysql02                  Mysql-MySQL_3306_Sort Scan                 31      22411.0  0.000000e+00```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import datetime
import pytz

# Reuse cached metric_df and thresholds_df from kernel
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

try:
    thresholds_df
except NameError:
    raise RuntimeError("thresholds_df not found in kernel. Run threshold computation step first.")

# Candidate components
components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat04","Tomcat03",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# 1) Compute incident window timestamps (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime.datetime(2021, 3, 25, 0, 0, 0))
end_local   = tz.localize(datetime.datetime(2021, 3, 25, 0, 30, 0))
# Convert to UTC epoch seconds
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

# 2) Filter metric_df to candidate components and incident window (inclusive)
metric_df["value"] = pd.to_numeric(metric_df["value"], errors="coerce")
metrics_incident_df = metric_df[
    (metric_df["cmdb_id"].isin(components)) &
    (metric_df["timestamp"] >= start_ts) &
    (metric_df["timestamp"] <= end_ts)
].copy()

# Cache metrics_incident_df in kernel (variable already assigned)

# 3) Join with thresholds_df on (cmdb_id, kpi_name)
# Ensure thresholds_df columns exist and numeric
thresholds_df["p95_value"] = pd.to_numeric(thresholds_df["p95_value"], errors="coerce")
thresholds_df["p5_value"]  = pd.to_numeric(thresholds_df["p5_value"], errors="coerce")
thresholds_df["count_full_series"] = thresholds_df["count_full_series"].astype(int)

merged = metrics_incident_df.merge(
    thresholds_df,
    on=["cmdb_id", "kpi_name"],
    how="inner",
    suffixes=("", "_th")
).copy()

# If no rows in merged, create empty anomalies table
if merged.shape[0] == 0:
    metric_anomalies_df = pd.DataFrame(
        columns=["cmdb_id","kpi_name","count_full_series","p95_value","p5_value",
                 "count_exceed_p95","earliest_exceed_ts","max_value_in_window",
                 "count_below_p5","earliest_below_ts","min_value_in_window"]
    )
    metric_anomalies_df_top20 = metric_anomalies_df.copy()
    total_groups = 0
else:
    # Group and compute aggregates
    merged["value"] = pd.to_numeric(merged["value"], errors="coerce")
    grp = merged.groupby(["cmdb_id", "kpi_name"], sort=False)

    # Basic aggregates from thresholds and window
    agg_basic = grp.agg(
        count_full_series = ("count_full_series", "first"),
        p95_value = ("p95_value", "first"),
        p5_value  = ("p5_value", "first"),
        max_value_in_window = ("value", "max"),
        min_value_in_window = ("value", "min")
    )

    # Compute counts and earliest timestamps relative to thresholds
    def cnt_exceed(g):
        return int((g["value"] >= g["p95_value"]).sum())

    def earliest_exceed(g):
        mask = g["value"] >= g["p95_value"]
        if mask.any():
            return int(g.loc[mask, "timestamp"].min())
        return pd.NA

    def cnt_below(g):
        return int((g["value"] <= g["p5_value"]).sum())

    def earliest_below(g):
        mask = g["value"] <= g["p5_value"]
        if mask.any():
            return int(g.loc[mask, "timestamp"].min())
        return pd.NA

    count_exceed_series = grp.apply(cnt_exceed).rename("count_exceed_p95")
    earliest_exceed_series = grp.apply(earliest_exceed).rename("earliest_exceed_ts")
    count_below_series = grp.apply(cnt_below).rename("count_below_p5")
    earliest_below_series = grp.apply(earliest_below).rename("earliest_below_ts")

    # Combine all into one DataFrame
    metric_anomalies_df = (
        agg_basic
        .join(count_exceed_series)
        .join(earliest_exceed_series)
        .join(count_below_series)
        .join(earliest_below_series)
        .reset_index()
    )

    # Ensure column order and types
    metric_anomalies_df = metric_anomalies_df[
        ["cmdb_id","kpi_name","count_full_series","p95_value","p5_value",
         "count_exceed_p95","earliest_exceed_ts","max_value_in_window",
         "count_below_p5","earliest_below_ts","min_value_in_window"]
    ]

    # Compute a sorting key and get top 20 by total anomaly counts
    metric_anomalies_df["anomaly_count_sum"] = metric_anomalies_df["count_exceed_p95"] + metric_anomalies_df["count_below_p5"]
    metric_anomalies_df = metric_anomalies_df.sort_values("anomaly_count_sum", ascending=False).reset_index(drop=True)
    metric_anomalies_df = metric_anomalies_df.drop(columns=["anomaly_count_sum"])

    metric_anomalies_df_top20 = metric_anomalies_df.head(20)
    total_groups = int(metric_anomalies_df.shape[0])

# Cache metric_anomalies_df and metrics_incident_df in kernel (already assigned)
metric_anomalies_df_top20, total_groups

```
Out[4]:
```
```python
# Summarize the anomaly aggregation results in plain English (uses cached variables)
import pytz, datetime

# Expect these variables in kernel: metric_anomalies_df_top20, metric_anomalies_df, metrics_incident_df, total_groups
try:
    metric_anomalies_df_top20
    metric_anomalies_df
    metrics_incident_df
    total_groups
except NameError:
    raise RuntimeError("Expected cached variables not found. Run previous steps first.")

tz = pytz.timezone('Asia/Shanghai')

def ts_to_local(ts):
    try:
        return tz.localize(datetime.datetime.fromtimestamp(int(ts))).strftime("%Y-03-25 %H:%M:%S %Z")
    except:
        return "N/A"

# Build compact summary
lines = []
lines.append(f"Incident-window anomaly summary (compact):")
lines.append(f"- Distinct (cmdb_id, kpi_name) groups in window: {total_groups}")
lines.append(f"- Cached variables: metrics_incident_df (filtered metrics) and metric_anomalies_df (anomaly aggregates).")
lines.append("")
lines.append("Top observations from the top-20 groups (by anomaly count):")
if metric_anomalies_df_top20.shape[0] == 0:
    lines.append("- No groups present in the incident window.")
else:
    # show up to 5 concise examples
    n_examples = min(5, metric_anomalies_df_top20.shape[0])
    for i, row in metric_anomalies_df_top20.head(n_examples).reset_index(drop=True).iterrows():
        ce = int(row["count_exceed_p95"])
        cb = int(row["count_below_p5"])
        cfull = int(row["count_full_series"])
        p95 = row["p95_value"]
        p5 = row["p5_value"]
        ee = row["earliest_exceed_ts"]
        eb = row["earliest_below_ts"]
        ee_local = ts_to_local(ee) if ee is not pd.NA else "NULL"
        eb_local = ts_to_local(eb) if eb is not pd.NA else "NULL"
        lines.append(
            f"  - [{i+1}] {row['cmdb_id']} | {row['kpi_name']}\n"
            f"     count_full_series={cfull}, p95={p95}, p5={p5}\n"
            f"     count_exceed_p95={ce}, earliest_exceed_ts={ee} ({ee_local})\n"
            f"     max_value_in_window={row['max_value_in_window']}\n"
            f"     count_below_p5={cb}, earliest_below_ts={eb} ({eb_local})\n"
            f"     min_value_in_window={row['min_value_in_window']}"
        )
    lines.append("")
    lines.append("Interpretation notes:")
    lines.append("- Many top groups have count_full_series = 31 (the 00:00–00:30 inclusive minutes), and several show count_exceed_p95 = 31 and count_below_p5 = 31 with p95 == p5 == value.")
    lines.append("  This indicates constant series values across the window (thresholds equal the observed values), so every point both meets >=P95 and <=P5 in those cases.")
    lines.append("- The top groups in the sample are from components such as IG01, Tomcat02, Tomcat03, and MG01.")
    lines.append("- Earliest exceed/below timestamps in the top rows are numeric epoch values (e.g., 1616601600) corresponding to the start of the window (2021-03-25 00:00:00 UTC+8).")
    lines.append("")
    lines.append("Stored objects for next steps: 'metrics_incident_df' (filtered incident data) and 'metric_anomalies_df' (full anomalies table).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count_full_series     p95_value      p5_value  count_exceed_p95  earliest_exceed_ts  max_value_in_window  count_below_p5  earliest_below_ts  min_value_in_window
0       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                 31  5.846119e+07  5.846119e+07                31          1616601600         5.846119e+07              31         1616601600         5.846119e+07
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                 31  2.040136e+09  2.040136e+09                31          1616601600         2.040136e+09              31         1616601600         2.040136e+09
2   Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                 31  5.000000e+02  5.000000e+02                31          1616601600         5.000000e+02              31         1616601600         5.000000e+02
3   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
4   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
5   Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
6   Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
7   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                 31  1.093000e+03  1.093000e+03                31          1616601600         1.093000e+03              31         1616601600         1.093000e+03
8   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...                 31  4.016560e+05  4.016560e+05                31          1616601600         4.016560e+05              31         1616601600         4.016560e+05
9   Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
10  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
11  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
12  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
13  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
14  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
15      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                 31  2.040136e+09  2.040136e+09                31          1616601600         2.040136e+09              31         1616601600         2.040136e+09
16      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                 31  5.889838e+07  5.889838e+07                31          1616601600         5.889838e+07              31         1616601600         5.889838e+07
17  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
18  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00
19  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                 31  0.000000e+00  0.000000e+00                31          1616601600         0.000000e+00              31         1616601600         0.000000e+00, 1707)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached variables: metric_anomalies_df, metrics_incident_df
try:
    metric_anomalies_df
    metrics_incident_df
except NameError:
    raise RuntimeError("Expected cached variables 'metric_anomalies_df' and 'metrics_incident_df' not found.")

# Ensure numeric columns
metric_anomalies_df["p95_value"] = pd.to_numeric(metric_anomalies_df["p95_value"], errors="coerce")
metric_anomalies_df["p5_value"]  = pd.to_numeric(metric_anomalies_df["p5_value"], errors="coerce")
metric_anomalies_df["max_value_in_window"] = pd.to_numeric(metric_anomalies_df["max_value_in_window"], errors="coerce")
metric_anomalies_df["min_value_in_window"] = pd.to_numeric(metric_anomalies_df["min_value_in_window"], errors="coerce")

# Compute severities
eps = 1e-9
metric_anomalies_df["up_severity"] = (
    (metric_anomalies_df["max_value_in_window"] - metric_anomalies_df["p95_value"])
    / metric_anomalies_df["p95_value"].abs().clip(lower=eps)
)
metric_anomalies_df["down_severity"] = (
    (metric_anomalies_df["p5_value"] - metric_anomalies_df["min_value_in_window"])
    / metric_anomalies_df["p5_value"].abs().clip(lower=eps)
)

# Select groups where either severity > 0.5
candidates_df = metric_anomalies_df[
    (metric_anomalies_df["up_severity"] > 0.5) | (metric_anomalies_df["down_severity"] > 0.5)
].copy()

# For each candidate, determine breach_type (the larger severity) and earliest breach timestamp/value
def find_earliest_breach(row):
    cm = row["cmdb_id"]
    kp = row["kpi_name"]
    p95 = row["p95_value"]
    p5  = row["p5_value"]
    up = row["up_severity"]
    down = row["down_severity"]
    # choose breach type by larger severity (tie -> prefer UP)
    if up >= down:
        btype = "UP"
        mask = (metrics_incident_df["cmdb_id"] == cm) & (metrics_incident_df["kpi_name"] == kp) & (metrics_incident_df["value"] >= p95)
    else:
        btype = "DOWN"
        mask = (metrics_incident_df["cmdb_id"] == cm) & (metrics_incident_df["kpi_name"] == kp) & (metrics_incident_df["value"] <= p5)
    subset = metrics_incident_df.loc[mask, ["timestamp", "value"]]
    if subset.shape[0] == 0:
        return pd.Series([btype, pd.NA, pd.NA])
    earliest_idx = subset["timestamp"].idxmin()
    earliest_ts = int(subset.loc[earliest_idx, "timestamp"])
    earliest_val = float(subset.loc[earliest_idx, "value"])
    return pd.Series([btype, earliest_ts, earliest_val])

if not candidates_df.empty:
    eb = candidates_df.apply(find_earliest_breach, axis=1)
    eb.columns = ["breach_type", "earliest_breach_ts", "earliest_breach_value"]
    candidates_df = pd.concat([candidates_df.reset_index(drop=True), eb.reset_index(drop=True)], axis=1)
else:
    # empty structure
    candidates_df["breach_type"] = pd.Series(dtype=str)
    candidates_df["earliest_breach_ts"] = pd.Series(dtype="object")
    candidates_df["earliest_breach_value"] = pd.Series(dtype="float64")

# Compute sort key and prepare final columns
candidates_df["max_severity"] = candidates_df[["up_severity", "down_severity"]].max(axis=1)

cols_out = [
    "cmdb_id","kpi_name","count_full_series","p95_value","p5_value",
    "up_severity","down_severity","breach_type","earliest_breach_ts","earliest_breach_value"
]

severe_metric_candidates_df = candidates_df.sort_values("max_severity", ascending=False).reset_index(drop=True)
severe_metric_candidates_df_top20 = severe_metric_candidates_df[cols_out].head(20)

# Total number of candidate groups found
total_candidates = int(severe_metric_candidates_df.shape[0])

# Cache severe_metric_candidates_df in kernel (variable assigned)
severe_metric_candidates_df_top20, total_candidates

```
Out[5]:
```
- Total candidate groups found: 104.

- High-level verdict:
  - Most selected groups show UP breaches (value above the global P95). Only UP breaches appear in the top results.
  - Many breaches start at the very beginning of the incident window (2021-03-25 00:00:00 UTC+8), indicating persistent/constant exceedance through the window.

- Top examples (sorted by severity):
  1) Mysql01 | Mysql-MySQL_3306_max trx lock memory bytes
     - count_full_series=31, p95=0.0, p5=0.0
     - up_severity ≈ 1.136e+12 (very large), breach_type=UP
     - earliest_breach_ts = 1616601600 → 2021-03-25 00:00:00 (UTC+8)
     - earliest_breach_value = 1136.0
  2) Mysql01 | Mysql-MySQL_3306_Max trx rows locked
     - count_full_series=31, p95=0.0, p5=0.0
     - up_severity ≈ 1.0e+09, breach_type=UP
     - earliest_breach_ts = 1616601600 → 2021-03-25 00:00:00 (UTC+8)
     - earliest_breach_value = 1.0
  3) Mysql01 | Mysql-MySQL_3306_MaxTrxRowsModified
     - similar characteristics to #2, earliest breach 00:00:00 UTC+8
  4) MG01 | OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
     - count_full_series=31, p95=0.0, p5=0.0, up_severity ~1.0e+09, earliest 00:00:00
  5) Redis02 | OSLinux-CPU_CPU_CPUWio
     - count_full_series=31, p95≈0.2187, p5≈0.00418
     - up_severity ≈ 3.59, breach_type=UP
     - earliest_breach_ts = 1616601660 → 2021-03-25 00:01:00 (UTC+8)
     - earliest_breach_value ≈ 1.0039
  6) MG02 | OSLinux-CPU_CPU_CPUWio
     - count_full_series=30, p95≈1.6120, p5≈0.02079
     - up_severity ≈ 3.02, earliest 2021-03-25 00:00:00 (UTC+8), earliest value ≈ 6.4879
  - (Other top entries include several OSLinux disk/CPU metrics for MG01/MG02/Tomcat nodes with up_severity > 0.5; many have earliest breach at 00:00:00 UTC+8.)

- Interpretation notes:
  - Extremely large up_severity values for some MySQL metrics occur where p95 (and p5) are zero while observed window values are non-zero—this yields very large relative breach scores.
  - Several OS-level metrics (CPU wait, disk read/rtps/bytes) show >50% breaches and are good candidates for follow-up.
  - Earliest breach times clustered at 00:00:00 UTC+8 imply the anomaly was present at the start of the incident window.

- Next steps / artifact:
  - The detailed candidate table is cached as severe_metric_candidates_df for further investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count_full_series  p95_value  p5_value   up_severity  down_severity breach_type  earliest_breach_ts  earliest_breach_value
0    Mysql01         Mysql-MySQL_3306_max trx lock memory bytes                 31   0.000000  0.000000  1.136000e+12       0.000000          UP          1616601600            1136.000000
1    Mysql01               Mysql-MySQL_3306_Max trx rows locked                 31   0.000000  0.000000  1.000000e+09       0.000000          UP          1616601600               1.000000
2    Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified                 31   0.000000  0.000000  1.000000e+09       0.000000          UP          1616601600               1.000000
3       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                 31   0.000000  0.000000  1.000000e+09       0.000000          UP          1616601600               0.000000
4    Redis02                             OSLinux-CPU_CPU_CPUWio                 31   0.218707  0.004183  3.590149e+00       0.004422          UP          1616601660               1.003900
5       MG02                             OSLinux-CPU_CPU_CPUWio                 30   1.612048  0.020790  3.024633e+00       0.201539          UP          1616601600               6.487900
6       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead                 16   7.866667  0.000000  3.000000e+00       0.000000          UP          1616601600              31.466667
7       MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps                 16   0.983333  0.000000  3.000000e+00       0.000000          UP          1616601600               3.933333
8   Tomcat03                    OSLinux-CPU_CPU-2_SingleCpuidle                 16  16.850864  0.000000  2.964500e+00       0.000000          UP          1616602440              66.805255
9   Tomcat04                             OSLinux-CPU_CPU_CPUWio                 30   1.061913  0.022663  2.650770e+00       0.267543          UP          1616601600               3.876800
10  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                 16  96.533333  0.000000  2.407459e+00       0.000000          UP          1616601960             328.933333
11   Mysql01                         OSLinux-CPU_CPU_CPUCpuUtil                 19   4.627590  1.017580  2.347617e+00       0.407909          UP          1616601660              15.491400
12      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps                 15   0.060000  0.000000  2.333333e+00       0.000000          UP          1616601600               0.200000
13      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps                 15   0.850000  0.000000  2.333333e+00       0.000000          UP          1616601600               2.833333
14      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead                 15   6.800000  0.000000  2.333333e+00       0.000000          UP          1616601600              22.666667
15      MG01                    OSLinux-CPU_CPU-2_SingleCpuidle                 15   0.219300  0.000000  2.333333e+00       0.000000          UP          1616601600               0.731000
16  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                 15   0.300000  0.000000  2.333333e+00       0.000000          UP          1616602320               1.000000
17  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                 15   0.600000  0.000000  2.333333e+00       0.000000          UP          1616602080               2.000000
18      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead                 15  13.800000  0.000000  2.333333e+00       0.000000          UP          1616601600              46.000000
19  Tomcat04  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                 15   0.090000  0.000000  2.333333e+00       0.000000          UP          1616601840               0.300000, 104)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse cached severe_metric_candidates_df
try:
    severe_metric_candidates_df
except NameError:
    raise RuntimeError("severe_metric_candidates_df not found in kernel. Run previous step first.")

# Ensure numeric severities and earliest_breach_ts are proper types
severe_metric_candidates_df["up_severity"] = pd.to_numeric(severe_metric_candidates_df["up_severity"], errors="coerce")
severe_metric_candidates_df["down_severity"] = pd.to_numeric(severe_metric_candidates_df["down_severity"], errors="coerce")
# earliest_breach_ts may contain pd.NA; convert to numeric where possible
severe_metric_candidates_df["earliest_breach_ts_num"] = pd.to_numeric(severe_metric_candidates_df["earliest_breach_ts"], errors="coerce")

# Compute per-component aggregates
def component_aggregate(df):
    # severe_kpi_count: distinct kpi_name
    severe_kpi_count = int(df["kpi_name"].nunique())
    # max_severity: max of max(up, down) per KPI => equivalently max of up/down across rows
    max_severity = float(df[["up_severity", "down_severity"]].max(axis=1).max())
    # earliest_breach_ts: minimum of earliest_breach_ts_num ignoring NaN
    if df["earliest_breach_ts_num"].dropna().size > 0:
        earliest_breach_ts = int(df["earliest_breach_ts_num"].dropna().min())
    else:
        earliest_breach_ts = pd.NA
    # example_kpis: take up to 5 distinct kpi_name examples ordered by per-KPI severity desc
    df = df.copy()
    df["kpi_max_sev"] = df[["up_severity", "down_severity"]].max(axis=1)
    # sort by kpi_max_sev desc, then take unique kpi_name preserving order
    df = df.sort_values("kpi_max_sev", ascending=False)
    seen = set()
    examples = []
    for name in df["kpi_name"].tolist():
        if name not in seen:
            seen.add(name)
            examples.append(name)
        if len(examples) >= 5:
            break
    example_kpis = ", ".join(examples)
    return pd.Series({
        "severe_kpi_count": severe_kpi_count,
        "max_severity": max_severity,
        "earliest_breach_ts": earliest_breach_ts,
        "example_kpis": example_kpis
    })

severe_by_component_df = (
    severe_metric_candidates_df
    .groupby("cmdb_id", sort=False)
    .apply(component_aggregate)
    .reset_index()
)

# Sort by severe_kpi_count descending and keep top 20
severe_by_component_df = severe_by_component_df.sort_values("severe_kpi_count", ascending=False).reset_index(drop=True)
severe_by_component_df_top20 = severe_by_component_df[["cmdb_id","severe_kpi_count","max_severity","earliest_breach_ts","example_kpis"]].head(20)

# Total number of distinct components found
total_components = int(severe_by_component_df["cmdb_id"].nunique())

# Cache severe_by_component_df in kernel (variable already assigned)
severe_by_component_df_top20, total_components

```
Out[6]:
```
```python
# Summarize the per-component severity results in plain English using cached severe_by_component_df_top20 and total_components
try:
    severe_by_component_df_top20
    total_components
except NameError:
    raise RuntimeError("Expected cached variables not found. Run previous steps first.")

# Build concise summary string
lines = []
lines.append(f"Per-component severe summary (compact):")
lines.append(f"- Total distinct components with severe KPIs: {total_components}")
lines.append("")
lines.append("Top components by number of severe KPIs (severe_kpi_count, max_severity, earliest_breach_ts):")
for _, r in severe_by_component_df_top20.iterrows():
    comp = r["cmdb_id"]
    cnt = int(r["severe_kpi_count"])
    maxsev = r["max_severity"]
    ets = r["earliest_breach_ts"]
    examples = r["example_kpis"]
    lines.append(f"- {comp}: {cnt} KPIs, max_severity={maxsev}, earliest_breach_ts={ets}; example_kpis={examples}")

lines.append("")
lines.append("Key notes:")
lines.append("- MG01 has the largest number of severe KPIs (12). MG02 and Tomcat04 also show many affected KPIs (11 each).")
lines.append("- Mysql01 and Mysql02 appear with several severe MySQL metrics; Mysql01 shows an extremely large max_severity due to p95/p5 being zero (relative ratio becomes very large).")
lines.append("- Many components have earliest_breach_ts = 1616601600 (the start of the incident window), indicating anomalies present at the window start.")
lines.append("- The table 'severe_by_component_df' is cached for the next step and lists up to 5 example KPIs per component ordered by severity.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  severe_kpi_count  max_severity  earliest_breach_ts                                       example_kpis
0       MG01                12  1.000000e+09          1616601600  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT, ...
1       MG02                11  3.024633e+00          1616601600  OSLinux-CPU_CPU_CPUWio, OSLinux-OSLinux_LOCALD...
2   Tomcat04                11  2.650770e+00          1616601600  OSLinux-CPU_CPU_CPUWio, OSLinux-OSLinux_LOCALD...
3    Mysql01                10  1.136000e+12          1616601600  Mysql-MySQL_3306_max trx lock memory bytes, My...
4    Mysql02                10  1.000000e+00          1616601660  Mysql-MySQL_3306_Max trx rows locked, Mysql-My...
5   Tomcat01                10  2.407459e+00          1616601660  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...
6   Tomcat02                 8  1.500000e+00          1616601660  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...
7   Tomcat03                 8  2.964500e+00          1616601600  OSLinux-CPU_CPU-2_SingleCpuidle, OSLinux-CPU_C...
8   apache01                 7  1.000000e+00          1616602740  OSLinux-CPU_CPU-1_SingleCpuidle, OSLinux-OSLin...
9       IG01                 6  1.222222e+00          1616601660  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps...
10      IG02                 6  8.181818e-01          1616601660  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTp...
11   Redis02                 3  3.590149e+00          1616601660  OSLinux-CPU_CPU_CPUWio, OSLinux-CPU_CPU_CPUCpu...
12   Redis01                 1  9.444438e-01          1616602140                    OSLinux-CPU_CPU-2_SingleCpuUtil
13  apache02                 1  5.793669e-01          1616601660                    OSLinux-CPU_CPU-3_SingleCpuUtil, 14)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import datetime
import pytz

# Reuse cached trace_df if available, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Candidate components
components = [
    "apache01","apache02","Tomcat01","Tomcat02","Tomcat04","Tomcat03",
    "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"
]

# Ensure numeric
trace_df["value"] = pd.to_numeric(trace_df["value"], errors="coerce")

# 1) Compute global thresholds (count_full_series and p95_value) for each (cmdb_id, trace_name)
trace_candidates = trace_df[trace_df["cmdb_id"].isin(components)].copy()

gb = trace_candidates.groupby(["cmdb_id", "trace_name"], sort=False)

count_df = gb["value"].count().reset_index(name="count_full_series")
p95_df = (
    gb["value"]
    .apply(lambda x: float(np.nanpercentile(x.dropna(), 95)) if x.dropna().size>0 else np.nan)
    .reset_index(name="p95_value")
)

trace_thresholds_df = count_df.merge(p95_df, on=["cmdb_id", "trace_name"])
# cache trace_thresholds_df in kernel

# 2) Compute incident window timestamps (UTC+8) and filter trace_df
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime.datetime(2021, 3, 25, 0, 0, 0))
end_local   = tz.localize(datetime.datetime(2021, 3, 25, 0, 30, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

traces_incident_df = trace_df[
    (trace_df["cmdb_id"].isin(components)) &
    (trace_df["timestamp"] >= start_ts) &
    (trace_df["timestamp"] <= end_ts)
].copy()
# cache traces_incident_df in kernel

# 3) Join and compute per-group anomaly aggregates within the incident window
merged = traces_incident_df.merge(trace_thresholds_df, on=["cmdb_id", "trace_name"], how="inner")

if merged.shape[0] == 0:
    trace_anomalies_df = pd.DataFrame(columns=[
        "cmdb_id","trace_name","count_full_series","p95_value",
        "count_exceed_p95","earliest_exceed_ts","max_value_in_window"
    ])
else:
    merged["value"] = pd.to_numeric(merged["value"], errors="coerce")
    grp = merged.groupby(["cmdb_id", "trace_name"], sort=False)

    agg_basic = grp.agg(
        count_full_series = ("count_full_series", "first"),
        p95_value = ("p95_value", "first"),
        max_value_in_window = ("value", "max")
    )

    def cnt_exceed(g):
        return int((g["value"] >= g["p95_value"]).sum())

    def earliest_exceed(g):
        mask = g["value"] >= g["p95_value"]
        if mask.any():
            return int(g.loc[mask, "timestamp"].min())
        return pd.NA

    count_exceed_series = grp.apply(cnt_exceed).rename("count_exceed_p95")
    earliest_exceed_series = grp.apply(earliest_exceed).rename("earliest_exceed_ts")

    trace_anomalies_df = (
        agg_basic
        .join(count_exceed_series)
        .join(earliest_exceed_series)
        .reset_index()
    )

# 4) Prepare compact top-20 table and total group count; cache trace_anomalies_df
trace_anomalies_df = trace_anomalies_df[[
    "cmdb_id","trace_name","count_full_series","p95_value",
    "count_exceed_p95","earliest_exceed_ts","max_value_in_window"
]]

trace_anomalies_df = trace_anomalies_df.sort_values("count_exceed_p95", ascending=False).reset_index(drop=True)
trace_anomalies_df_top20 = trace_anomalies_df.head(20)

total_groups = int(trace_anomalies_df.shape[0])

# Return compact results (cached variables: trace_thresholds_df, traces_incident_df, trace_anomalies_df)
trace_anomalies_df_top20, total_groups

```
Out[7]:
```
Summary of trace-feature anomaly detection (compact):

- Total distinct (cmdb_id, trace_name) groups present in the incident window: 189.

- High-level finding: IG01 dominates the top anomalies. Many IG01 trace features (row_count and duration variants) exceeded their global P95 within the 00:00–00:30 incident window.

- Top examples (sorted by count_exceed_p95):
  1) IG01 | trace.to_Tomcat03.row_count
     - count_full_series=31, p95_value=79.0, count_exceed_p95=3, earliest_exceed_ts=1616601660, max_value_in_window=98.0
  2) Tomcat03 | trace.from_IG01.row_count
     - count_full_series=31, p95_value=79.0, count_exceed_p95=3, earliest_exceed_ts=1616601660, max_value_in_window=98.0
  3) IG01 | trace.self.duration_p95
     - count_full_series=31, p95_value=1.695625, count_exceed_p95=2, earliest_exceed_ts=1616601660, max_value_in_window=3.053

- Other notable patterns:
  - Several IG01 → Tomcat* traces show 2 exceedances each (duration_p95, duration_mean, row_count).
  - Many earliest_exceed_ts values are 1616601600 or 1616601660 (start/early points in the window).
  - The largest count_full_series values are typically 30–31, so exceedances represent a small number of minutes but are repeatable across multiple trace features.

- Artifacts cached for next steps: trace_thresholds_df (global p95/count), traces_incident_df (filtered window), and trace_anomalies_df (joined anomaly aggregates).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                       trace_name  count_full_series   p95_value  count_exceed_p95  earliest_exceed_ts  max_value_in_window
0       IG01      trace.to_Tomcat03.row_count                 31   79.000000                 3          1616601660            98.000000
1   Tomcat03        trace.from_IG01.row_count                 31   79.000000                 3          1616601660            98.000000
2       IG01          trace.self.duration_p95                 31    1.695625                 2          1616601660             3.053000
3       IG01  trace.to_Tomcat01.duration_mean                 30    0.465524                 2          1616601600             0.555520
4       IG01             trace.self.row_count                 31  606.000000                 2          1616601660           734.000000
5       IG01   trace.to_Tomcat01.duration_p95                 30    2.307802                 2          1616601720             4.439700
6       IG01      trace.to_Tomcat01.row_count                 30   71.850000                 2          1616601660            96.000000
7       IG01   trace.to_Tomcat02.duration_p95                 31    1.723200                 2          1616601720             2.236500
8       IG01  trace.to_Tomcat02.duration_mean                 31    0.331865                 2          1616602080             0.444548
9       IG01      trace.to_Tomcat02.row_count                 31   76.500000                 2          1616601660            89.000000
10      IG01  trace.to_Tomcat03.duration_mean                 31    0.410445                 2          1616601660             0.431265
11      IG01   trace.to_Tomcat03.duration_p95                 31    1.656300                 2          1616601600             2.661200
12      IG01         trace.self.duration_mean                 31    0.375121                 2          1616601600             0.470646
13      IG01   trace.to_Tomcat04.duration_p95                 30    2.009045                 2          1616601600             4.335500
14      IG01      trace.to_Tomcat04.row_count                 30   80.750000                 2          1616601660            95.000000
15      IG02         trace.self.duration_mean                 31    0.321331                 2          1616601720             0.559168
16      IG02          trace.self.duration_p95                 31    1.547500                 2          1616601660             3.127000
17      IG02             trace.self.row_count                 31  584.000000                 2          1616601660           780.000000
18      IG02  trace.to_Tomcat01.duration_mean                 30    0.398877                 2          1616601720             0.437087
19      IG02   trace.to_Tomcat01.duration_p95                 30    2.271970                 2          1616601720             2.933800, 189)```
```

In [ ]:
# In[8]:

import re
import pandas as pd

# Reuse cached variables
try:
    trace_anomalies_df
except NameError:
    raise RuntimeError("trace_anomalies_df not found in kernel. Run previous steps first.")
try:
    severe_by_component_df
except NameError:
    raise RuntimeError("severe_by_component_df not found in kernel. Run previous steps first.")

# Work on a copy
df = trace_anomalies_df.copy()

# Extract direction and counterpart component when trace_name contains 'to_' or 'from_'
# Pattern captures 'to'/'from' and the following component token (alphanumeric)
pattern = r'\b(to|from)_([A-Za-z0-9]+)'
extracted = df['trace_name'].str.extract(pattern, expand=True)
extracted.columns = ['dir_token', 'counterpart']

# Attach parsed fields
df['direction'] = extracted['dir_token'].str.upper()  # 'TO' or 'FROM' or NaN
df['counterpart_cmdb'] = extracted['counterpart']

# Keep only rows that had a counterpart parsed
pairs_df = df[df['counterpart_cmdb'].notna()].copy()
pairs_df['origin_cmdb'] = pairs_df['cmdb_id']

# Keep only rows where either origin or counterpart is in severe components list
severe_components = set(severe_by_component_df['cmdb_id'].unique())
mask_involves_severe = pairs_df['origin_cmdb'].isin(severe_components) | pairs_df['counterpart_cmdb'].isin(severe_components)
trace_pairs_df = pairs_df[mask_involves_severe].copy()

# Select and order required columns
trace_pairs_df = trace_pairs_df[
    ["origin_cmdb", "counterpart_cmdb", "direction", "trace_name",
     "count_full_series", "p95_value", "count_exceed_p95", "earliest_exceed_ts", "max_value_in_window"]
].reset_index(drop=True)

# Sort by count_exceed_p95 desc, earliest_exceed_ts asc (NaNs last)
trace_pairs_df_sorted = trace_pairs_df.sort_values(
    by=["count_exceed_p95", "earliest_exceed_ts"],
    ascending=[False, True],
    na_position='last'
).reset_index(drop=True)

# Top 20 compact view
trace_pairs_df_top20 = trace_pairs_df_sorted.head(20)

# Aggregate summaries: counts per counterpart_cmdb (top 10) and per origin_cmdb (top 10)
counterpart_counts = trace_pairs_df.groupby("counterpart_cmdb").size().reset_index(name="pair_count")
counterpart_counts_top10 = counterpart_counts.sort_values("pair_count", ascending=False).head(10).reset_index(drop=True)

origin_counts = trace_pairs_df.groupby("origin_cmdb").size().reset_index(name="pair_count")
origin_counts_top10 = origin_counts.sort_values("pair_count", ascending=False).head(10).reset_index(drop=True)

# Cache trace_pairs_df for next step (full table)
# trace_pairs_df variable already assigned

# Return compact outputs
trace_pairs_df_top20, counterpart_counts_top10, origin_counts_top10

```
Out[8]:
```
- Top anomalous trace pairs (compact):
  - Highest-ranked pair: IG01 → Tomcat03 (trace.to_Tomcat03.row_count) and the reciprocal Tomcat03 ← IG01 (trace.from_IG01.row_count). Each has count_full_series=31, count_exceed_p95=3, earliest_exceed_ts=1616601660, max_value_in_window=98.0.
  - Other frequent pairs involve IG01 → Tomcat01/02/03/04 (duration and row_count features) with count_exceed_p95=2 and earliest_exceed_ts often 1616601600.
  - MG01 and MG02 also appear as origins communicating with Tomcat nodes and docker nodes (e.g., MG01 → Tomcat01/03/04, MG01 → dockerA1/dockerB1; MG02 → Tomcat01/02/03/dockerB1).

- Aggregate counts (compact):
  - Top counterpart_cmdb by number of anomalous trace pairs: root (21), IG02 (12), MG01 (12), MG02 (12), IG01 (12), Tomcat01 (12), Tomcat02 (12), Tomcat04 (12), Tomcat03 (12), dockerA1 (12).
  - Top origin_cmdb by number of anomalous trace pairs: MG02 (39), MG01 (39), IG02 (15), Tomcat01 (15), Tomcat03 (15), Tomcat02 (15), Tomcat04 (15), IG01 (12).

- Timing pattern:
  - Many earliest_exceed_ts values are 1616601600 (start of the incident window) or 1616601660 (one minute in), indicating breaches present at or immediately after window start.

- Interpretation / next focus:
  - Communication/trace anomalies are concentrated on interactions between IG components and Tomcat nodes, and heavy involvement of MG01/MG02 as origins. These components (IG01/IG02, MG01/MG02, Tomcat01–04) should be prioritized for investigation.
  - The reciprocal nature (IG01↔Tomcat03) suggests degraded request/response behavior between those services.

- Artifacts:
  - The parsed trace-pairs table is cached as trace_pairs_df for follow-up analysis.

The original code execution output of IPython Kernel is also provided below for reference:

(   origin_cmdb counterpart_cmdb direction                         trace_name  count_full_series  p95_value  count_exceed_p95  earliest_exceed_ts  max_value_in_window
0         IG01         Tomcat03        TO        trace.to_Tomcat03.row_count                 31  79.000000                 3          1616601660            98.000000
1     Tomcat03             IG01      FROM          trace.from_IG01.row_count                 31  79.000000                 3          1616601660            98.000000
2         IG01         Tomcat01        TO    trace.to_Tomcat01.duration_mean                 30   0.465524                 2          1616601600             0.555520
3         IG01         Tomcat03        TO     trace.to_Tomcat03.duration_p95                 31   1.656300                 2          1616601600             2.661200
4         IG01         Tomcat04        TO     trace.to_Tomcat04.duration_p95                 30   2.009045                 2          1616601600             4.335500
5         IG02         Tomcat03        TO    trace.to_Tomcat03.duration_mean                 31   0.386216                 2          1616601600             0.491706
6         IG02         Tomcat03        TO     trace.to_Tomcat03.duration_p95                 31   1.582575                 2          1616601600             2.753000
7         IG01         Tomcat04        TO    trace.to_Tomcat04.duration_mean                 30   0.362687                 2          1616601600             0.599398
8         MG01         Tomcat01      FROM  trace.from_Tomcat01.duration_mean                 30   0.399115                 2          1616601600             0.502644
9         MG01         Tomcat01      FROM   trace.from_Tomcat01.duration_p95                 30   2.394387                 2          1616601600             4.291200
10        MG01         Tomcat03      FROM  trace.from_Tomcat03.duration_mean                 31   0.277168                 2          1616601600             0.332196
11        MG01         Tomcat04      FROM  trace.from_Tomcat04.duration_mean                 30   0.176342                 2          1616601600             0.420706
12        MG01         dockerA1        TO    trace.to_dockerA1.duration_mean                 30   0.119518                 2          1616601600             0.244667
13        MG01         dockerA1        TO     trace.to_dockerA1.duration_p95                 30   0.970740                 2          1616601600             1.599000
14        MG01         dockerB1        TO    trace.to_dockerB1.duration_mean                 30   0.381941                 2          1616601600             0.746525
15        MG02         Tomcat01      FROM  trace.from_Tomcat01.duration_mean                 30   0.245131                 2          1616601600             0.314606
16        MG02         Tomcat02      FROM   trace.from_Tomcat02.duration_p95                 31   1.038075                 2          1616601600             1.087200
17        MG02         Tomcat03      FROM  trace.from_Tomcat03.duration_mean                 31   0.341089                 2          1616601600             0.444013
18        MG02         dockerB1      FROM  trace.from_dockerB1.duration_mean                 30   0.210874                 2          1616601600             0.564955
19        MG02         dockerB1      FROM   trace.from_dockerB1.duration_p95                 30   0.899280                 2          1616601600             3.965500,   counterpart_cmdb  pair_count
0             root          21
1             IG02          12
2             MG01          12
3             MG02          12
4             IG01          12
5         Tomcat01          12
6         Tomcat02          12
7         Tomcat04          12
8         Tomcat03          12
9         dockerA1          12,   origin_cmdb  pair_count
0        MG02          39
1        MG01          39
2        IG02          15
3    Tomcat01          15
4    Tomcat03          15
5    Tomcat02          15
6    Tomcat04          15
7        IG01          12)```
```

In [ ]:
# In[9]:

import pandas as pd
import re
import numpy as np
import pytz

# Reuse cached severe_metric_candidates_df; expect it to exist
try:
    severe_metric_candidates_df
except NameError:
    raise RuntimeError("severe_metric_candidates_df not found in kernel. Run previous steps first.")

# Components of interest
components_of_interest = [
    "MG01","MG02","IG01","IG02",
    "Tomcat01","Tomcat02","Tomcat03","Tomcat04",
    "Mysql01","Mysql02","Redis02"
]

# Work on a copy
df = severe_metric_candidates_df.copy()

# Ensure numeric columns for safety
df["up_severity"] = pd.to_numeric(df["up_severity"], errors="coerce").fillna(0.0)
df["down_severity"] = pd.to_numeric(df["down_severity"], errors="coerce").fillna(0.0)
df["earliest_breach_ts"] = pd.to_numeric(df["earliest_breach_ts"], errors="coerce")
df["earliest_breach_value"] = pd.to_numeric(df["earliest_breach_value"], errors="coerce")

# Filter to components of interest
comp_df = df[df["cmdb_id"].isin(components_of_interest)].copy()

# Helper: map a kpi_name to a single category label if possible
def map_kpi_to_category(kpi_name):
    s = kpi_name.upper()
    # Priority matching
    if any(token in s for token in ["CPU", "CPUTIME", "CPUWIO", "CPULOAD", "CPUCPU"]):
        return "CPU"
    if any(token in s for token in ["MEMORY", "HEAP", "NOHEAP", "HEAPMEMORY", "JVM_MEMORY", "JVM-MEMORY", "JVM"]):
        return "Memory/JVM"
    if any(token in s for token in ["LOCALDISK", "DSK", "DISK", "IO", "READ", "WRITE", "DSKREAD", "DSKRTPS", "DSKBPS", "DSKPER"]):
        return "Disk"
    if any(token in s for token in ["NETWORK", "TCP", "PACKET", "NET"]):
        return "Network"
    if any(token in s for token in ["MYSQL", "MYSQL-","MYSQL_","MYSQL_3306","MYSQL_3306_"]):
        return "MySQL"
    if any(token in s for token in ["LOG", "GC", "SESSION", "REQUEST", "THREAD"]):
        return "App/Log"
    # Fallback: try extract significant token
    tokens = re.split(r'[-_"\s\.]+', kpi_name)
    for t in tokens:
        ut = t.upper()
        if len(ut) >= 3:
            return ut[:12]  # short token
    return "OTHER"

# Build per-component aggregates
rows = []
for comp in components_of_interest:
    sub = comp_df[comp_df["cmdb_id"] == comp]
    if sub.shape[0] == 0:
        # Component has no severe KPIs present; include row with zeros/NA
        rows.append({
            "cmdb_id": comp,
            "earliest_breach_ts": pd.NA,
            "severe_kpi_count": 0,
            "dominant_kpi_categories": "",
            "top3_kpis": ""
        })
        continue

    # earliest breach ts
    if sub["earliest_breach_ts"].dropna().size > 0:
        earliest_ts = int(sub["earliest_breach_ts"].dropna().min())
    else:
        earliest_ts = pd.NA

    severe_kpi_count = int(sub["kpi_name"].nunique())

    # dominant_kpi_categories: count mapped categories across distinct kpi_names
    # Map each distinct kpi_name once
    kpi_names = sub["kpi_name"].dropna().astype(str).unique().tolist()
    cat_counts = {}
    for k in kpi_names:
        cat = map_kpi_to_category(k)
        cat_counts[cat] = cat_counts.get(cat, 0) + 1
    # sort categories by count desc
    cat_items = sorted(cat_counts.items(), key=lambda x: x[1], reverse=True)
    # format up to 3 token:count pairs
    dominant_kpi_categories = ", ".join(f"{k}:{v}" for k, v in cat_items[:3])

    # top3_kpis by severity (max of up/down)
    sub["max_sev"] = sub[["up_severity", "down_severity"]].max(axis=1)
    sub_sorted = sub.sort_values(["max_sev", "earliest_breach_ts"], ascending=[False, True])
    top3 = []
    for _, r in sub_sorted.head(3).iterrows():
        # determine breach type and severity value
        if r["up_severity"] >= r["down_severity"]:
            btype = "UP"
            sev = float(r["up_severity"])
        else:
            btype = "DOWN"
            sev = float(r["down_severity"])
        kpi_entry = f"{r['kpi_name']} ({btype}, sev={sev:.3g}, ts={r['earliest_breach_ts']}, val={r['earliest_breach_value']})"
        top3.append(kpi_entry)
    top3_kpis = " ; ".join(top3)

    rows.append({
        "cmdb_id": comp,
        "earliest_breach_ts": earliest_ts,
        "severe_kpi_count": severe_kpi_count,
        "dominant_kpi_categories": dominant_kpi_categories,
        "top3_kpis": top3_kpis
    })

component_diagnostics_df = pd.DataFrame(rows)

# Sort by earliest_breach_ts ascending (NA last), then severe_kpi_count desc
component_diagnostics_df["earliest_breach_ts_sort"] = pd.to_numeric(component_diagnostics_df["earliest_breach_ts"], errors="coerce")
component_diagnostics_df = component_diagnostics_df.sort_values(
    by=["earliest_breach_ts_sort", "severe_kpi_count"],
    ascending=[True, False],
    na_position='last'
).drop(columns=["earliest_breach_ts_sort"]).reset_index(drop=True)

# Cache component_diagnostics_df in kernel (variable assigned)
# Prepare compact view (one row per component)
component_diagnostics_df_view = component_diagnostics_df[[
    "cmdb_id","earliest_breach_ts","severe_kpi_count","dominant_kpi_categories","top3_kpis"
]]

total_components_returned = int(component_diagnostics_df_view.shape[0])

component_diagnostics_df_view, total_components_returned

```
Out[9]:
```
Summary of per-component diagnostics (compact):

- Total components returned: 11 (MG01, MG02, IG01, IG02, Tomcat01–04, Mysql01, Mysql02, Redis02).

- High-level finding:
  - Most components show their earliest breach at or immediately after 1616601600 (the start of the incident window) or 1616601660, indicating anomalies were present at the window start for many KPIs.
  - Disk and CPU-related KPIs are the dominant problem categories across many components. MySQL-related KPIs are prominent for Mysql01/Mysql02 and show extreme relative severity in some cases.

- Per-component highlights (one line each):
  - MG01 — earliest_breach_ts=1616601600; severe_kpi_count=12; dominant categories: Disk (8), CPU (3), Network (1). Example top KPIs include network TCP-FIN-WAIT and multiple disk I/O KPIs (high severities).
  - MG02 — earliest_breach_ts=1616601600; severe_kpi_count=11; dominant: Disk (9), CPU, Network. Top KPI examples: OSLinux CPU WIO and several local-disk metrics.
  - Tomcat04 — earliest_breach_ts=1616601600; severe_kpi_count=11; dominant: Disk (6), CPU (3), Network (2). Top KPI: OSLinux CPU WIO (UP, sev≈2.65).
  - Mysql01 — earliest_breach_ts=1616601600; severe_kpi_count=10; dominant: Disk (4), MySQL (3), CPU (2). Top KPI: Mysql-MySQL_3306_max trx lock memory bytes (UP, extremely large relative severity; earliest breach at 1616601600).
  - Tomcat03 — earliest_breach_ts=1616601600; severe_kpi_count=8; dominant: CPU (4), Disk (4). Top KPI: OSLinux-CPU-2_SingleCpuidle (UP, sev≈2.96).
  - Tomcat01 — earliest_breach_ts=1616601660; severe_kpi_count=10; dominant: Disk (7), CPU (2), Network (1). Top KPI: local-disk DSKRead (large UP severity).
  - Mysql02 — earliest_breach_ts=1616601660; severe_kpi_count=10; dominant: Disk (7), MySQL (3). Top KPI: Mysql Max trx rows locked (UP, notable severity).
  - Tomcat02 — earliest_breach_ts=1616601660; severe_kpi_count=8; dominant: Disk (5), Network (2), CPU (1). Top KPIs are disk-related.
  - IG01 — earliest_breach_ts=1616601660; severe_kpi_count=6; dominant: Disk (5), CPU (1). Top KPIs are local-disk metrics (sdb) and some trace/communication anomalies previously observed.
  - IG02 — earliest_breach_ts=1616601660; severe_kpi_count=6; dominant: Disk (6). Top KPIs are local-disk read/rtps metrics.
  - Redis02 — earliest_breach_ts=1616601660; severe_kpi_count=3; dominant: CPU (2), REDIS (1). Top KPI: OSLinux CPU WIO (UP, sev≈3.59).

- Priority for investigation (suggested):
  1. MG01 and MG02 — many severe disk I/O and CPU/WIO KPIs (highest counts).
  2. Tomcat cluster nodes (Tomcat01–04) — heavy disk and CPU signals, some with early breaches.
  3. Mysql01 and Mysql02 — MySQL metrics show high severities (Mysql01 has extreme relative breaches due to zero baseline p95/p5).
  4. IG01/IG02 and Redis02 — additional disk/CPU problems and communication anomalies (already observed in traces).

- Note on severity interpretation:
  - Some very large severity values arise where baseline p95/p5 are zero and observed values are nonzero (relative ratio becomes enormous). These should be interpreted as strong signals but checked against absolute impact (e.g., whether the metric meaningfully affects service behavior).

- Artifacts:
  - The component diagnostics table is cached as component_diagnostics_df for follow-up investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  earliest_breach_ts  severe_kpi_count   dominant_kpi_categories                                          top3_kpis
0       MG01          1616601600                12  Disk:8, CPU:3, Network:1  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT (...
1       MG02          1616601600                11  Disk:9, CPU:1, Network:1  OSLinux-CPU_CPU_CPUWio (UP, sev=3.02, ts=16166...
2   Tomcat04          1616601600                11  Disk:6, CPU:3, Network:2  OSLinux-CPU_CPU_CPUWio (UP, sev=2.65, ts=16166...
3    Mysql01          1616601600                10    Disk:4, MySQL:3, CPU:2  Mysql-MySQL_3306_max trx lock memory bytes (UP...
4   Tomcat03          1616601600                 8             CPU:4, Disk:4  OSLinux-CPU_CPU-2_SingleCpuidle (UP, sev=2.96,...
5   Tomcat01          1616601660                10  Disk:7, CPU:2, Network:1  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...
6    Mysql02          1616601660                10           Disk:7, MySQL:3  Mysql-MySQL_3306_Max trx rows locked (UP, sev=...
7   Tomcat02          1616601660                 8  Disk:5, Network:2, CPU:1  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...
8       IG01          1616601660                 6             Disk:5, CPU:1  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps...
9       IG02          1616601660                 6                    Disk:6  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTp...
10   Redis02          1616601660                 3            CPU:2, REDIS:1  OSLinux-CPU_CPU_CPUWio (UP, sev=3.59, ts=16166..., 11)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np

# Reuse cached variables from kernel
try:
    severe_metric_candidates_df
    trace_pairs_df
    component_diagnostics_df
    metric_anomalies_df
except NameError as e:
    raise RuntimeError("Expected cached variables not found in kernel: " + str(e))

# Ensure numeric and expected columns exist
for col in ["up_severity", "down_severity", "earliest_breach_ts", "earliest_breach_value"]:
    if col not in severe_metric_candidates_df.columns:
        severe_metric_candidates_df[col] = pd.NA
severe_metric_candidates_df["up_severity"] = pd.to_numeric(severe_metric_candidates_df["up_severity"], errors="coerce").fillna(0.0)
severe_metric_candidates_df["down_severity"] = pd.to_numeric(severe_metric_candidates_df["down_severity"], errors="coerce").fillna(0.0)

# Prepare helper to ensure max_value_in_window present by joining with metric_anomalies_df if needed
if "max_value_in_window" not in severe_metric_candidates_df.columns:
    if "max_value_in_window" in metric_anomalies_df.columns:
        sev_full = severe_metric_candidates_df.merge(
            metric_anomalies_df[["cmdb_id","kpi_name","max_value_in_window"]],
            on=["cmdb_id","kpi_name"], how="left"
        )
    else:
        severe_metric_candidates_df["max_value_in_window"] = pd.NA
        sev_full = severe_metric_candidates_df.copy()
else:
    sev_full = severe_metric_candidates_df.copy()

# Compute severity score for sorting
sev_full["severity_score"] = sev_full[["up_severity","down_severity"]].max(axis=1)

# Function to build evidence table for a component
def build_component_evidence(component, top_n=10):
    sub = sev_full[sev_full["cmdb_id"] == component].copy()
    if sub.shape[0] == 0:
        # empty frame with expected columns
        cols = ["kpi_name","breach_type","severity","earliest_breach_ts","earliest_breach_value","max_value_in_window"]
        empty = pd.DataFrame(columns=cols)
        # component-level summary from diagnostics table (if present)
        comp_row = component_diagnostics_df[component_diagnostics_df["cmdb_id"]==component]
        earliest = comp_row["earliest_breach_ts"].iat[0] if comp_row.shape[0]>0 else pd.NA
        skc = int(comp_row["severe_kpi_count"].iat[0]) if comp_row.shape[0]>0 else 0
        return empty, earliest, skc

    sub = sub.sort_values("severity_score", ascending=False).head(top_n)
    # determine breach_type and severity numeric
    def choose_breach(r):
        if r["up_severity"] >= r["down_severity"]:
            return pd.Series([ "UP", float(r["up_severity"]) ])
        else:
            return pd.Series([ "DOWN", float(r["down_severity"]) ])
    bt = sub.apply(choose_breach, axis=1)
    bt.columns = ["breach_type","severity"]
    out = pd.concat([sub.reset_index(drop=True), bt.reset_index(drop=True)], axis=1)
    # Select columns requested
    evidence = out[["kpi_name","breach_type","severity","earliest_breach_ts","earliest_breach_value","max_value_in_window"]].copy()
    # component-level summary from component_diagnostics_df
    comp_row = component_diagnostics_df[component_diagnostics_df["cmdb_id"]==component]
    earliest = comp_row["earliest_breach_ts"].iat[0] if comp_row.shape[0]>0 else pd.NA
    skc = int(comp_row["severe_kpi_count"].iat[0]) if comp_row.shape[0]>0 else int(sub["kpi_name"].nunique())
    return evidence, earliest, skc

# Build MG01 evidence
mg01_evidence_df, mg01_earliest_ts, mg01_severe_kpi_count = build_component_evidence("MG01", top_n=10)

# Build IG01 evidence
ig01_evidence_df, ig01_earliest_ts, ig01_severe_kpi_count = build_component_evidence("IG01", top_n=10)

# Build IG01 trace pairs (rows where origin or counterpart is IG01), limit top 20
tp = trace_pairs_df.copy()
ig_pairs_mask = (tp["origin_cmdb"] == "IG01") | (tp["counterpart_cmdb"] == "IG01")
ig01_trace_pairs_df = tp[ig_pairs_mask].copy()
# sort by count_exceed_p95 desc, earliest_exceed_ts asc (NaNs last)
ig01_trace_pairs_df = ig01_trace_pairs_df.sort_values(
    by=["count_exceed_p95","earliest_exceed_ts"], ascending=[False, True], na_position='last'
).reset_index(drop=True)
ig01_trace_pairs_df = ig01_trace_pairs_df.head(20)[
    ["origin_cmdb","counterpart_cmdb","direction","trace_name",
     "count_full_series","p95_value","count_exceed_p95","earliest_exceed_ts","max_value_in_window"]
].copy()

# Cache results in kernel variables for next step
mg01_evidence_df = mg01_evidence_df.reset_index(drop=True)
ig01_evidence_df = ig01_evidence_df.reset_index(drop=True)

# Also prepare compact component summary rows for MG01 and IG01
mg01_summary = pd.DataFrame([{
    "cmdb_id":"MG01",
    "earliest_breach_ts": mg01_earliest_ts,
    "severe_kpi_count": mg01_severe_kpi_count
}])
ig01_summary = pd.DataFrame([{
    "cmdb_id":"IG01",
    "earliest_breach_ts": ig01_earliest_ts,
    "severe_kpi_count": ig01_severe_kpi_count
}])

# Final displayed outputs (compact): evidence tables and IG01 trace pairs, plus component summaries
mg01_evidence_df, mg01_summary, ig01_evidence_df, ig01_summary, ig01_trace_pairs_df

```
Out[10]:
```
MG01 evidence
- Component summary: earliest_breach_ts=1616601600, severe_kpi_count=12.
- Top severe KPIs (up to 10, sorted by severity):
  1) OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — breach_type=UP, severity=1.0e+09, earliest_breach_ts=1616601600, earliest_breach_value=0.0, max_value_in_window=1.0  
  2) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead — UP, severity=3.0, earliest_breach_ts=1616601600, earliest_breach_value=31.466667, max=31.466667  
  3) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps — UP, severity=3.0, earliest_breach_ts=1616601600, earliest_breach_value=3.933333, max=3.933333  
  4) OSLinux-CPU_CPU-2_SingleCpuidle — UP, severity=2.333333, earliest_breach_ts=1616601600, earliest_breach_value=0.731000, max=0.731000  
  5) OSLinux-CPU_CPU_CPUWio — UP, severity≈1.143125, earliest_breach_ts=1616601600, earliest_breach_value=5.805800, max=5.805800  
  6) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps — UP, severity=1.0, earliest_breach_ts=1616601720, earliest_breach_value=7.000000, max=7.000000  
  7) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps — UP, severity=1.0, earliest_breach_ts=1616603040, earliest_breach_value=3.000000, max=3.000000  
  8) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPercent… — UP, severity=1.0, earliest_breach_ts=1616603160, earliest_breach_value=0.600000, max=0.600000  
  9) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead — UP, severity≈0.906328, earliest_breach_ts=1616602680, earliest_breach_value=177.733333, max=177.733333  
 10) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps — UP, severity≈0.849056, earliest_breach_ts=1616602680, earliest_breach_value=0.816667, max=0.816667

IG01 evidence
- Component summary: earliest_breach_ts=1616601660, severe_kpi_count=6.
- Top severe KPIs (up to 10; 6 present — sorted by severity):
  1) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps — UP, severity≈1.222222, earliest_breach_ts=1616601720, earliest_breach_value=1.0000, max=1.0000  
  2) OSLinux-CPU_CPU_CPUWio — UP, severity≈1.036340, earliest_breach_ts=1616601660, earliest_breach_value=2.0789, max=2.0789  
  3) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead — UP, severity≈0.818182, earliest_breach_ts=1616602080, earliest_breach_value=138.8000, max=138.8000  
  4) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps — UP, severity≈0.818182, earliest_breach_ts=1616602080, earliest_breach_value=1.6167, max=1.6167  
  5) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead… — UP, severity≈0.629914, earliest_breach_ts=1616601720, earliest_breach_value=85.0000, max=85.0000  
  6) OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPercent… — UP, severity≈0.538462, earliest_breach_ts=1616603280, earliest_breach_value=0.2000, max=0.2000

IG01 trace-pair anomalies (rows where origin or counterpart is IG01; top 20 shown)
- There are many IG01-related trace anomalies; top examples (sorted by count_exceed_p95 desc, earliest_exceed_ts asc):
  1) origin=IG01 → counterpart=Tomcat03, direction=TO, trace_name=trace.to_Tomcat03.row_count — count_full_series=31, p95=79.0, count_exceed_p95=3, earliest_exceed_ts=1616601660, max_value_in_window=98.0  
  2) origin=Tomcat03 → counterpart=IG01, direction=FROM, trace_name=trace.from_IG01.row_count — count_full_series=31, p95=79.0, count_exceed_p95=3, earliest_exceed_ts=1616601660, max=98.0  
  3) IG01 → Tomcat01, trace.to_Tomcat01.duration_mean — count_full_series=30, p95≈0.465524, count_exceed_p95=2, earliest_exceed_ts=1616601600, max≈0.555520  
  4) IG01 → Tomcat03, trace.to_Tomcat03.duration_p95 — count_full_series=31, p95≈1.656300, count_exceed_p95=2, earliest_exceed_ts=1616601600, max≈2.661200  
  5) IG01 → Tomcat04, trace.to_Tomcat04.duration_p95 — count_full_series=30, p95≈2.009045, count_exceed_p95=2, earliest_exceed_ts=1616601600, max≈4.335500  
  (Additional IG01 pairs include reciprocal from_IG01 traces on Tomcat01/03/04 and several row_count/duration pairs; many earliest_exceed_ts are 1616601600 or 1616601660.)

Caching
- The returned evidence tables are cached as:
  - mg01_evidence_df (MG01 KPI evidence)
  - ig01_evidence_df (IG01 KPI evidence)
  - ig01_trace_pairs_df (IG01-related trace pairs, top 20)

Concise interpretation
- MG01 shows many severe disk and CPU-related KPIs with earliest breaches at the start of the window (1616601600), including extreme relative breaches on TCP-FIN-WAIT and strong DSKRead/RTps signals — MG01 is a high-priority candidate.  
- IG01 shows disk and CPU issues plus multiple communication trace anomalies with Tomcat nodes (IG01↔Tomcat03 stands out); earliest IG01 breaches are at 1616601660.  
- Both components have multiple, high-severity signals and the IG01↔Tomcat communication pattern provides corroborating trace-level evidence linking IG01 to service-level degradation.

The original code execution output of IPython Kernel is also provided below for reference:

(                                            kpi_name breach_type breach_type      severity  earliest_breach_ts  earliest_breach_value  max_value_in_window
0       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          UP          UP  1.000000e+09          1616601600               0.000000             1.000000
1    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead          UP          UP  3.000000e+00          1616601600              31.466667            31.466667
2    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps          UP          UP  3.000000e+00          1616601600               3.933333             3.933333
3                    OSLinux-CPU_CPU-2_SingleCpuidle          UP          UP  2.333333e+00          1616601600               0.731000             0.731000
4                             OSLinux-CPU_CPU_CPUWio          UP          UP  1.143125e+00          1616601600               5.805800             5.805800
5     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps          UP          UP  1.000000e+00          1616601720               7.000000             7.000000
6     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps          UP          UP  1.000000e+00          1616603040               3.000000             3.000000
7  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...          UP          UP  1.000000e+00          1616603160               0.600000             0.600000
8    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead          UP          UP  9.063282e-01          1616602680             177.733333           177.733333
9    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps          UP          UP  8.490560e-01          1616602680               0.816667             0.816667,   cmdb_id  earliest_breach_ts  severe_kpi_count
0    MG01          1616601600                12,                                             kpi_name breach_type breach_type  severity  earliest_breach_ts  earliest_breach_value  max_value_in_window
0     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps          UP          UP  1.222222          1616601720                 1.0000               1.0000
1                             OSLinux-CPU_CPU_CPUWio          UP          UP  1.036340          1616601660                 2.0789               2.0789
2    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead          UP          UP  0.818182          1616602080               138.8000             138.8000
3    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps          UP          UP  0.818182          1616602080                 1.6167               1.6167
4  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...          UP          UP  0.629914          1616601720                85.0000              85.0000
5  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...          UP          UP  0.538462          1616603280                 0.2000               0.2000,   cmdb_id  earliest_breach_ts  severe_kpi_count
0    IG01          1616601660                 6,    origin_cmdb counterpart_cmdb direction                       trace_name  count_full_series  p95_value  count_exceed_p95  earliest_exceed_ts  max_value_in_window
0         IG01         Tomcat03        TO      trace.to_Tomcat03.row_count                 31  79.000000                 3          1616601660            98.000000
1     Tomcat03             IG01      FROM        trace.from_IG01.row_count                 31  79.000000                 3          1616601660            98.000000
2         IG01         Tomcat01        TO  trace.to_Tomcat01.duration_mean                 30   0.465524                 2          1616601600             0.555520
3         IG01         Tomcat03        TO   trace.to_Tomcat03.duration_p95                 31   1.656300                 2          1616601600             2.661200
4         IG01         Tomcat04        TO   trace.to_Tomcat04.duration_p95                 30   2.009045                 2          1616601600             4.335500
5         IG01         Tomcat04        TO  trace.to_Tomcat04.duration_mean                 30   0.362687                 2          1616601600             0.599398
6     Tomcat01             IG01      FROM    trace.from_IG01.duration_mean                 30   0.465524                 2          1616601600             0.555520
7     Tomcat03             IG01      FROM     trace.from_IG01.duration_p95                 31   1.656300                 2          1616601600             2.661200
8     Tomcat04             IG01      FROM    trace.from_IG01.duration_mean                 30   0.362687                 2          1616601600             0.599398
9     Tomcat04             IG01      FROM     trace.from_IG01.duration_p95                 30   2.009045                 2          1616601600             4.335500
10        IG01         Tomcat01        TO      trace.to_Tomcat01.row_count                 30  71.850000                 2          1616601660            96.000000
11        IG01         Tomcat02        TO      trace.to_Tomcat02.row_count                 31  76.500000                 2          1616601660            89.000000
12        IG01         Tomcat03        TO  trace.to_Tomcat03.duration_mean                 31   0.410445                 2          1616601660             0.431265
13        IG01         Tomcat04        TO      trace.to_Tomcat04.row_count                 30  80.750000                 2          1616601660            95.000000
14    Tomcat01             IG01      FROM        trace.from_IG01.row_count                 30  71.850000                 2          1616601660            96.000000
15    Tomcat02             IG01      FROM        trace.from_IG01.row_count                 31  76.500000                 2          1616601660            89.000000
16    Tomcat03             IG01      FROM    trace.from_IG01.duration_mean                 31   0.410445                 2          1616601660             0.431265
17    Tomcat04             IG01      FROM        trace.from_IG01.row_count                 30  80.750000                 2          1616601660            95.000000
18        IG01         Tomcat01        TO   trace.to_Tomcat01.duration_p95                 30   2.307802                 2          1616601720             4.439700
19        IG01         Tomcat02        TO   trace.to_Tomcat02.duration_p95                 31   1.723200                 2          1616601720             2.236500)```
```